# HandwrittenOCR - استخراج وتصحيح نصوص الخط اليدوي

دفتر Google Colab للعمل مع المشروع.

---

## الخلية 1: تثبيت المكتبات

In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q pdf2image easyocr pyspellchecker langdetect pytesseract \
    transformers torch torchvision Pillow opencv-python-headless \
    pandas ar-corrector ipywidgets

## الخلية 2: إعداد المشروع والمسارات

In [ ]:
import sys, os

# ربط Google Drive
from google.colab import drive
drive.mount('/content/drive')

# إعداد المسارات
PDF_PATH = '/content/drive/MyDrive/python notes.pdf'
OUTPUT_DIR = '/content/drive/MyDrive/Handwriting_Dataset'

# استيراد وتشغيل التطبيق
from config import Config
from src.logger import setup_logging
from src.correction import init_correctors
from src.recognition import OCREngine
from src.database import HandwritingDB
from src.pdf_processor import PDFProcessor
from src.review_ui import ReviewUI

# إنشاء الإعدادات
config = Config.from_colab_drive(pdf_name='python notes.pdf')
config.ensure_dirs()

# إعداد التسجيل
logger = setup_logging(config)
logger.info('بدء تشغيل المشروع في Colab')

## الخلية 3: تحميل النماذج

In [ ]:
import time

print('جاري تحميل النماذج...')
start = time.time()

# تحميل المدققات الإملائية
init_correctors()

# تحميل محرك التعرف
ocr_engine = OCREngine(
    trocr_model_name=config.trocr_model_name,
    ocr_languages=config.ocr_languages,
    max_text_length=config.max_text_length,
)

logger.info(f'تم التحميل في {time.time()-start:.2f} ثانية')
print(f'تم التحميل في {time.time()-start:.2f} ثانية')

## الخلية 4: معالجة PDF

In [ ]:
# تعديل نطاق الصفحات حسب الحاجة
config.pages_start = 1
config.pages_end = 2

# تهيئة قاعدة البيانات ومعالج PDF
db = HandwritingDB(config.db_path)
processor = PDFProcessor(config, ocr_engine, db)

# تشغيل المعالجة
stats = processor.process()

# عرض الإحصائيات
print(f"\nالصفحات: {stats['pages_processed']}")
print(f"الكلمات: {stats['total_words']}")
print(f"إخفاقات OCR: {stats['ocr_failures']}")
print(f"تصحيحات: {stats['spell_corrections']}")
print(f"الوقت: {stats['time_seconds']:.2f} ثانية")

## الخلية 5: واجهة المراجعة والتصحيح

In [ ]:
review_ui = ReviewUI(db, config.feedback_csv)
review_ui.launch()

## الخلية 6: ملفات المراقبة

In [ ]:
print('ملفات المراقبة والتطوير:')
print(f'  سجل الأحداث:     {config.log_file}')
print(f'  إحصائيات:        {config.stats_json}')
print(f'  تصحيحات:         {config.feedback_csv}')
print(f'  قاعدة البيانات:  {config.db_path}')